# Assignment #1: Pseudonymisation Techniques and Considerations
- Dataset: Crossfit [Daset](https://data.world/bgadoci/crossfit-data) (In this assignment only the athletes file was used) 
- Credits: Dataset was put together by Sam Swift
- ToDo: To run the jupyter notebook the requirements.txt need be installed (`pip install -r requirements.txt`)

In [40]:
import pandas as pd

# Read csv as dataframe
dataframe = pd.read_csv("athletes.csv")

C:\Users\Julius\AppData\Local\Temp\ipykernel_332\1944461800.py:4: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv("athletes.csv")


## 3.1 Pseudonymisation
Goals:
-  Determine which attributes qualify as explicit personally identifiable information (3.1.1)
    - Why? 
    - What method was used to identify these? 
- Generate pseudonymous values for the identified attributes (3.1.2)

### 3.1.1 Identifying Attributes containing Personally Identifiable Information (PII)
- In order to identify th attributes first a better understanding of the structure of the dataset needs to be obtained
    - What columns does the dataset contain and in what format are the attribute values?
        - Therefore, each column and the first value of each column (which is not empty or Null) is printed

- By inspecting the different columns and the data format, several attributes which have the potential to contain explicit personally identifiable information can be identified
    - `athelete_id`
        -  This really depends on the usage of this id! Considerations to take into account are: 
            - Is the `athlete_id` only used as an internal id of this dataset or does it maybe even refer to an official id?
            - Are there other datasets available which may have a similar source to this dataset? Thus, these other datasets may use the same `athlete_id`
    - `name`
        - The name allows to identify an individual
    - `team`
        - Depending on the size of the team, this could allow to identify a specific athlete
    - `affiliate` 
        - Depending on the affiliate and the amount of contracted athletes, this could allow to identify an individual
    - All stats of the athletes
        - If an athlete has really remarkable stats (maybe even a world record in a category), this could allow to identify the individual
    - `train` 
        - If an athlete has a special and famous training routine, this could allow to identify him
    - `background`
        - If an athlete has a famous background or mentions names, this could allow to identify him
    - `experience`
        - If an athlete mentions concrete information about his experience (e.g. name of current coach), this could allow to identify him

-> As can be seen, all columns could potentially contain outliers which could be then used to identify an individual. As agreed on, for each of the columns (only those holding numbers or names) potentially leaking PII one anonymization technique was chosen which keeps the data loss as minimal as possible without leaking obvious PII. It is important to note, that there could still be columns containing PII. However, this chance for outliers was accepted in this assignment. Columns containing descriptions (e.g. train, background and experience) where to anonymized and thus the chance for outliers accepted.

### 3.1.2 Pseudonymizing
- For pseudonymisation especially the name is suitable, because it can be easily replaced with a fake name, without being as obvious or destroying the purpose of the dataset
- With the help of the `anonymizedf` library new fake names can be created for the `name` column

In [41]:
from anonymizedf import anonymizedf

# Prepare the data to be anonymized
an = anonymizedf.anonymize(dataframe)

In [42]:
# Add new column with fake name
an.fake_names("name")

# Drop old original name column
dataframe = dataframe.drop(columns=['name'])

# Rename Fake_name column in name
dataframe = dataframe.rename(columns={'Fake_name': 'name'})

print(dataframe.iloc[:4])

   athlete_id               region          team             affiliate gender  \
0      2554.0           South West   Double Edge  Double Edge CrossFit   Male   
1      3517.0                  NaN           NaN                   NaN   Male   
2      4691.0                  NaN           NaN                   NaN    NaN   
3      5164.0  Southern California  LAX CrossFit          LAX CrossFit   Male   

    age  height  weight   fran  helen  ...  backsq  pullups  \
0  24.0    70.0   166.0    NaN    NaN  ...   301.0      NaN   
1  42.0    70.0   190.0    NaN    NaN  ...     NaN      NaN   
2   NaN     NaN     NaN    NaN    NaN  ...     NaN      NaN   
3  40.0    67.0     NaN  211.0  788.0  ...   323.0     84.0   

                                    eat  \
0                                   NaN   
1                                   NaN   
2                                   NaN   
3  I eat 1-3 full cheat meals per week|   

                                               train  \
0  I w

## 3.2 Randomisation
Goal:
- Use randomisation technique to generate random strings (no meaning) (3.2.1)
- Use randomisation technique to generate random but meaningful replacements (3.2.2)
- Create a lookup table to keep track of the changes (3.2.3)

### 3.2.1 Random Strings
1. Identify columns to replace with random strings
    - Here columns containing words can be used 
        - `name`
        - `region`
        - `team`
        - `affiliate` 
2. Generate a function which is responsible for generating random values for replacement
3. Overwrite each value in respective attributes by using the generated value

In [43]:
def get_first_not_not_empty_value(df_column):
    return df_column.dropna().iloc[0] if not df_column.dropna().empty else None

# Iterate each column 
for column in dataframe.columns:
    first_value = get_first_not_not_empty_value(dataframe[column])
    print(f"Column: '{column}', Example Data: {first_value}")

Column: 'athlete_id', Example Data: 2554.0
Column: 'region', Example Data: South West
Column: 'team', Example Data: Double Edge
Column: 'affiliate', Example Data: Double Edge CrossFit
Column: 'gender', Example Data: Male
Column: 'age', Example Data: 24.0
Column: 'height', Example Data: 70.0
Column: 'weight', Example Data: 166.0
Column: 'fran', Example Data: 211.0
Column: 'helen', Example Data: 788.0
Column: 'grace', Example Data: 228.0
Column: 'filthy50', Example Data: 1053.0
Column: 'fgonebad', Example Data: 55.0
Column: 'run400', Example Data: 106.0
Column: 'run5k', Example Data: 1000.0
Column: 'candj', Example Data: 178.0
Column: 'snatch', Example Data: 183.0
Column: 'deadlift', Example Data: 427.0
Column: 'backsq', Example Data: 301.0
Column: 'pullups', Example Data: 84.0
Column: 'eat', Example Data: I eat 1-3 full cheat meals per week|
Column: 'train', Example Data: I workout mostly at a CrossFit Affiliate|I have a coach who determines my programming|I record my workouts|
Column: 

In [44]:
import random
import string

# Define function to create random string
def generate_random_string(length):
    letters = string.ascii_letters
    return ''.join(random.choice(letters) for _ in range(length))

# Apply randomization on specified columns
dataframe['name'] = dataframe['name'].apply(lambda x: generate_random_string(10))
dataframe['region'] = dataframe['region'].apply(lambda x: generate_random_string(15))
dataframe['team'] = dataframe['team'].apply(lambda x: generate_random_string(20))
dataframe['affiliate'] = dataframe['affiliate'].apply(lambda x: generate_random_string(25))

print(dataframe.iloc[:4])

   athlete_id           region                  team  \
0      2554.0  vwOrFPJwCBryRAj  DHPPKgnIbjUQaHhImCrv   
1      3517.0  hdXPYLoCwkdrLuc  HASvBXJPzsxLFXyewXid   
2      4691.0  RUIJOQREutZYgAu  btBGWrKkSIvyKSJkeZAa   
3      5164.0  WpIYrIyHPztubwm  jdLdptXRYNaRiZrljLoa   

                   affiliate gender   age  height  weight   fran  helen  ...  \
0  ihKvQnBtUmfaIHdszreBGNVfK   Male  24.0    70.0   166.0    NaN    NaN  ...   
1  gZaIPsZeRxtwOxFcOVxBOXmPw   Male  42.0    70.0   190.0    NaN    NaN  ...   
2  PAuSeEZMMEaXFAgNuevQkdKAQ    NaN   NaN     NaN     NaN    NaN    NaN  ...   
3  rvDqEckQZwwdXfzVlfwGENRux   Male  40.0    67.0     NaN  211.0  788.0  ...   

   backsq  pullups                                   eat  \
0   301.0      NaN                                   NaN   
1     NaN      NaN                                   NaN   
2     NaN      NaN                                   NaN   
3   323.0     84.0  I eat 1-3 full cheat meals per week|   

                 

### 3.2.2 Randomization with meaningful strings
1. Identify columns suitable
    - `name`
        -  For randomizing the name, the first letter will be kept equal to the original in order to keep similarities
2. Write functionality to randomize the values

In [45]:
from anonymizedf import anonymizedf

# To see the changes properly we have to reload the initial data
dataframe = pd.read_csv("athletes.csv")
        
# Randomization the name
# insert your functionality Emirkan


# Interchange values for region, team and affiliate
def interchange_values(column_name):
    dataframe[column_name] = dataframe[column_name].sample(frac=1).reset_index(drop=True)
        
interchange_values('region')
interchange_values('team')
interchange_values('affiliate')

print(dataframe.iloc[:4])

C:\Users\Julius\AppData\Local\Temp\ipykernel_332\3969017026.py:4: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv("athletes.csv")


   athlete_id           name      region                team  \
0      2554.0      Pj Ablang         NaN                 NaN   
1      3517.0  Derek Abdella         NaN  CrossFit Ridgewood   
2      4691.0            NaN         NaN                 NaN   
3      5164.0    Abo Brandon  North East                 NaN   

            affiliate gender   age  height  weight   fran  ...  deadlift  \
0                 NaN   Male  24.0    70.0   166.0    NaN  ...     427.0   
1                 NaN   Male  42.0    70.0   190.0    NaN  ...       NaN   
2     CrossFit Murbah    NaN   NaN     NaN     NaN    NaN  ...       NaN   
3  Guerrilla CrossFit   Male  40.0    67.0     NaN  211.0  ...     359.0   

   backsq  pullups                                   eat  \
0   301.0      NaN                                   NaN   
1     NaN      NaN                                   NaN   
2     NaN      NaN                                   NaN   
3   323.0     84.0  I eat 1-3 full cheat meals per week|  

## 3.3 Aggregation
Goal:
- Determine attributes which qualify for aggregation (3.3.1)
- Write a function to perform that aggregation process (3.3.2)

### 3.3.1 Determine Attributes for Aggregation

In [47]:
# Print the each column with its first not empty value
for column in dataframe.columns:
    first_value = get_first_not_not_empty_value(dataframe[column])
    print(f"Column: '{column}', Example Data: {first_value}")

Column: 'athlete_id', Example Data: 2554.0
Column: 'name', Example Data: Pj Ablang
Column: 'region', Example Data: South East
Column: 'team', Example Data: CFOSLO WOLFPACK
Column: 'affiliate', Example Data: Border City CrossFit
Column: 'gender', Example Data: Male
Column: 'age', Example Data: 24.0
Column: 'height', Example Data: 70.0
Column: 'weight', Example Data: 166.0
Column: 'fran', Example Data: 211.0
Column: 'helen', Example Data: 788.0
Column: 'grace', Example Data: 228.0
Column: 'filthy50', Example Data: 1053.0
Column: 'fgonebad', Example Data: 55.0
Column: 'run400', Example Data: 106.0
Column: 'run5k', Example Data: 1000.0
Column: 'candj', Example Data: 178.0
Column: 'snatch', Example Data: 183.0
Column: 'deadlift', Example Data: 427.0
Column: 'backsq', Example Data: 301.0
Column: 'pullups', Example Data: 84.0
Column: 'eat', Example Data: I eat 1-3 full cheat meals per week|
Column: 'train', Example Data: I workout mostly at a CrossFit Affiliate|I have a coach who determines m

- Numerical attributes can be easily aggregated
- Especially those related to information about the individual are suitable for aggregation
    - `age`
    - `height` 
    - `weight`
- To be able to identify different levels of values, the extremes have to be identified 

In [48]:
print(f"Minimum age': {dataframe['age'].nsmallest(5)}, Maximum age: {dataframe['age'].nlargest(5)}")
print(f"Minimum height': {dataframe['height'].nsmallest(10)}, Maximum height: {dataframe['height'].nlargest(10)}")
print(f"Minimum weight': {dataframe['weight'].nsmallest(10)}, Maximum weight: {dataframe['weight'].nlargest(10)}")

Minimum age': 7257      13.0
276479    13.0
331209    13.0
347070    13.0
267865    14.0
Name: age, dtype: float64, Maximum age: 27115     125.0
363267    125.0
358673    124.0
364750    124.0
404768    124.0
Name: age, dtype: float64
Minimum height': 6157     0.0
7384     0.0
10847    0.0
17122    0.0
19320    0.0
19665    0.0
23203    0.0
25896    0.0
30426    0.0
31051    0.0
Name: height, dtype: float64, Maximum height: 67446     8388607.0
361003       7087.0
382676        787.0
29607         748.0
122814        744.0
374476        740.0
84035         732.0
110804        732.0
117237        732.0
132676        732.0
Name: height, dtype: float64
Minimum weight': 29311     1.0
68995     1.0
70012     1.0
123238    1.0
135247    1.0
139870    1.0
139914    1.0
139950    1.0
153348    1.0
170930    1.0
Name: weight, dtype: float64, Maximum weight: 377897    20175.0
124868     9008.0
152034     2441.0
27925      2215.0
6157       2205.0
121387     2113.0
256935     2000.0
51636      180

- Because even by observing ten extremes for the height and weight the values didn't make sense, in the following average values for the entire population were taken

In [49]:
bins_age = [0, 18, 30, 45, 60, 100]
labels_age = ['0-18', '19-30', '31-45', '46-60', '60+']

bins_height = [0, 159, 169, 179, 189, 199, 220]
labels_height = ['0-159', '160-169', '170-179', '180-189', '190-199', '200+']

bins_weight = [0, 49, 59, 69, 79, 89, 99, 130]
labels_weight = ['0-49', '50-59', '60-69', '70-79', '80-89', '90-99', '100+']

dataframe['age'] = pd.cut(dataframe['age'], bins=bins_age, labels=labels_age)
dataframe['height'] = pd.cut(dataframe['height'], bins=bins_age, labels=labels_age)
dataframe['weight'] = pd.cut(dataframe['weight'], bins=bins_age, labels=labels_age)

print(dataframe)

        athlete_id             name         region                   team  \
0           2554.0        Pj Ablang     South East                    NaN   
1           3517.0    Derek Abdella      Australia        CFOSLO WOLFPACK   
2           4691.0              NaN            NaN                    NaN   
3           5164.0      Abo Brandon      Australia                    NaN   
4           5286.0      Bryce Abbey  South Central              Team DRiV   
...            ...              ...            ...                    ...   
423001    574489.0       Odo Renata            NaN                    NaN   
423002    585696.0    Lozzie Trevor   Mid Atlantic                    NaN   
423003    608828.0    Marisol Smith   Central East                    NaN   
423004    628881.0  Pedrini Morgane            NaN  Beast Mode Guerrillas   
423005    631847.0       Eden Kenny      Australia  CrossFit James Island   

                        affiliate  gender    age height weight   fran  ... 

## 3.4 Perturbation
Goal:
- Select attributes to add noise to (3.4.1)
- Implement the functionality to add the noise (3.4.2)
    - The original distribution of values should be preserved 
- Analyse distribution of original data and then with noise added (3.4.3)

### 3.4.1 Select attributes
- Similar to the aggregation, perturbation as well fits best for numbers as attribute values 
- Because the stats of the athletes are the main subject of the dataset they should normally be no noise added to them
    - Even more, the correctness of the value really matters in order to be able to compare different athletes. Here already a small noise is not good
- However, as already mentioned in 3.1 this kind of information loss is accepted in this assignment
- Therefore, the stats will be pertubated 

### 3.4.2 Pertubate Values
- Standard deviation was used to determine noise

In [50]:
import pandas as pd
import numpy as np

# Load data from the CSV file
df = pd.read_csv('athletes.csv')

columns_to_perturb = ['pullups', 'helen', 'grace', 'filthy50', 'fgonebad', 'run400', 'run5k', 'candj', 'snatch', 'deadlift', 'backsq']

# Analyze the non-missing numeric values before perturbation
stats_before_perturbation = {}
for column in columns_to_perturb:
    data = df[column].values
    non_missing_values = [value for value in data if not pd.isna(value)]
    mean_before = np.mean(non_missing_values)
    std_dev_before = np.std(non_missing_values)
    stats_before_perturbation[column] = {'mean_before': mean_before, 'std_dev_before': std_dev_before}

print("Before Perturbation:")
for column, stats in stats_before_perturbation.items():
    print(f"Mean {column} (Before): {stats['mean_before']}")
    print(f"Std Dev {column} (Before): {stats['std_dev_before']}")

# Choose a noise factor
noise_factor = 40  # Adjust this based on your privacy requirements

# Perturb the non-missing numeric values for each specified column
for column in columns_to_perturb:
    perturbed_data = []
    for value in df[column]:
        if not pd.isna(value):
            perturbed_value = value + np.random.normal(0, noise_factor)
            perturbed_value = int(abs(round(perturbed_value)))
            perturbed_data.append(perturbed_value)
        else:
            perturbed_data.append(np.nan)
    df[column] = perturbed_data  # Update the column in the DataFrame with perturbed data

# Analyze the non-missing numeric values after perturbation
stats_after_perturbation = {}
for column in columns_to_perturb:
    non_missing_values_after = [value for value in df[column] if not pd.isna(value)]
    mean_after = np.mean(non_missing_values_after)
    std_dev_after = np.std(non_missing_values_after)
    stats_after_perturbation[column] = {'mean_after': mean_after, 'std_dev_after': std_dev_after}

print("\nAfter Perturbation:")
for column, stats in stats_after_perturbation.items():
    print(f"Mean {column} (After): {stats['mean_after']}")
    print(f"Std Dev {column} (After): {stats['std_dev_after']}")


# Save the perturbed data back to a CSV file without scientific notation
df.to_csv('athletes.csv', index=False, float_format='%.0f')

C:\Users\Julius\AppData\Local\Temp\ipykernel_332\1141632013.py:5: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('athletes.csv')


Before Perturbation:
Mean pullups (Before): 42720.95178627885
Std Dev pullups (Before): 9545983.344212158
Mean helen (Before): 1208.0773473364377
Std Dev helen (Before): 68240.1019186712
Mean grace (Before): 577.1168486930911
Std Dev grace (Before): 48910.41159161189
Mean filthy50 (Before): 2128.3887080944264
Std Dev filthy50 (Before): 60548.334745471846
Mean fgonebad (Before): 1476.9643553702333
Std Dev fgonebad (Before): 97625.2341837532
Mean run400 (Before): 529.9320327249843
Std Dev run400 (Before): 56287.20315347341
Mean run5k (Before): 3411.343740477048
Std Dev run5k (Before): 125196.3842495196
Mean candj (Before): 272.8280078517738
Std Dev candj (Before): 25968.934922050452
Mean snatch (Before): 244.9627261513158
Std Dev snatch (Before): 27089.575656127934
Mean deadlift (Before): 698.3312869072084
Std Dev deadlift (Before): 55232.26891560957
Mean backsq (Before): 586.7361763348625
Std Dev backsq (Before): 50529.395095568274

After Perturbation:
Mean pullups (After): 42730.087081

## 3.5 Data Analysis
Goal:
- Determine data loss using function (3.5.1)
- Discuss pros and cons (3.5.2)
- attributes included in the information loss function: ['pullups', 'helen', 'grace', 'filthy50', 'fgonebad', 'run400', 'run5k', 'candj', 'snatch', 'deadlift', 'backsq'] + maybe aggregated values: age, height, weight


In [51]:
import pandas as pd
import numpy as np

# Read the CSV files
file1 = 'athletes.csv' 
file2 = 'athletes_og.csv'  

data1 = pd.read_csv(file1)
data2 = pd.read_csv(file2)

# Columns to calculate information loss
columns_to_compare = ['pullups', 'helen', 'grace', 'filthy50', 'fgonebad', 'run400', 'run5k', 'candj', 'snatch', 'deadlift', 'backsq']

# Calculate Information Loss
def calculate_information_loss(x, y, S):
    return (x - y) / np.sqrt(2 * S)

total_information_loss = 0
n = len(data1)  # Assuming both datasets have the same number of rows
p = len(columns_to_compare)  # Number of columns to compare

for column in columns_to_compare:
    x = data1[column]
    y = data2[column]
    S = np.mean((x - y) ** 2)  # Variance of the differences in the column

    information_loss = np.sum(calculate_information_loss(x, y, S)) / (p * n)
    total_information_loss += information_loss

print("Total Information Loss:", total_information_loss)


C:\Users\Julius\AppData\Local\Temp\ipykernel_332\2320207545.py:8: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  data1 = pd.read_csv(file1)
C:\Users\Julius\AppData\Local\Temp\ipykernel_332\2320207545.py:9: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  data2 = pd.read_csv(file2)


Total Information Loss: 0.008134947382432842
